# DocAgent: Gemma 4 E2B and 12B (Ollama Local)

This notebook runs `gemma4:e2b` and `gemma4:12b` on **Ollama Local** (`local=True`). It does not call Ollama Cloud, GPT, DeepSeek, or an LLM judge.

- Required inputs in the JupyterLab working directory: `input_gt_rule.txt` with 408 GT rules and `SPECIFICATION_ALL.txt`.
- Both models use the exact same 80,000-character chunks (1,500-character overlap), generated from the full specification, as the previous 31B Cloud notebook.
- Automatic resume saves each completed chunk. If interrupted, rerun the notebook with the same configuration to reuse completed chunks.
- Results are written separately under `results/docagent/gemma4_e2b/` and `results/docagent/gemma4_12b/`.
- The notebook performs generation only. Semantic matching and correctness validation are deliberately not performed.

Before running, ensure the Ollama Local service is active and **both model tags are available locally**. Model execution may require substantial RAM or VRAM. `MODEL_SETTINGS[model]['context']` is the allocated Ollama context size, not the maximum advertised model context. Use the current settings first; changing them during a partially completed run causes a new experiment configuration.

The base output limit matches the previous Cloud run. On an output-length failure, only that chunk is retried with a larger output budget, recorded in its checkpoint. This is a recovery mechanism, not a change to the generation prompt.

## 1. Configuration

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import hashlib
import json
import os
import re

RUN_MODELS = [
    'gemma4:e2b',
    'gemma4:12b',
]
MODEL_SETTINGS = {
    'gemma4:e2b': {'local': True, 'context': 65536},
    'gemma4:12b': {'local': True, 'context': 65536},
}
OLLAMA_LOCAL_HOST = 'http://127.0.0.1:11434'
TEMPERATURE = 1
MAX_OUTPUT_TOKENS = 8192
MAX_RETRY_OUTPUT_TOKENS = 24576
LLM_ATTEMPTS = 3
MAX_PROMPT_CHARS = 2000000
CHUNK_CHARS = 80000
CHUNK_OVERLAP_CHARS = 1500
EXPECTED_TARGETS = 8
EXPECTED_GT = 408
GT_FILE = Path('input_gt_rule.txt')
AUTO_RESUME = True
RESUME_SUITE_DIRS = {}

assert RUN_MODELS and len(set(RUN_MODELS)) == len(RUN_MODELS)
assert all(model in MODEL_SETTINGS for model in RUN_MODELS)
assert all(MODEL_SETTINGS[model]['local'] is True for model in RUN_MODELS)
assert GT_FILE.is_file(), f'Missing ground truth: {GT_FILE}'
assert 0 <= CHUNK_OVERLAP_CHARS < CHUNK_CHARS
assert 0 < MAX_OUTPUT_TOKENS <= MAX_RETRY_OUTPUT_TOKENS

clients = {}

def call_llm(model, prompt, max_output_tokens=MAX_OUTPUT_TOKENS):
    from ollama import Client
    if not MODEL_SETTINGS[model]['local']:
        raise ValueError(f'Only local models are allowed: {model}')
    if 'local' not in clients:
        clients['local'] = Client(host=OLLAMA_LOCAL_HOST, timeout=1800)
    response = clients['local'].chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={
            'temperature': TEMPERATURE,
            'num_ctx': MODEL_SETTINGS[model]['context'],
            'num_predict': max_output_tokens,
        },
    )
    return (
        (response.message.content or '').strip(),
        getattr(response, 'done_reason', None),
    )

## 2. Discover targets and validate GT counts

In [2]:
def parse_target_file(text):
    targets = {}
    current = None
    metadata = re.compile(r'^(?:FORMAT|NUMBER|MODE|WIDTH|FIELDS|EXTENSIONS)\s*:', re.I)
    for line_number, raw in enumerate(text.splitlines(), 1):
        line = raw.strip()
        if not line:
            continue
        header = re.fullmatch(r'(INSTRUCTION|REGISTER)\s*:\s*([A-Za-z0-9_.]+)', line, re.I)
        if header:
            name = header.group(2).lower()
            if name in targets:
                raise ValueError(f'Duplicate target: {name}')
            current = {'name': name, 'type': 'instruction' if header.group(1).upper() == 'INSTRUCTION' else 'csr', 'rules': {}}
            targets[name] = current
            continue
        if current is None:
            raise ValueError(f'Missing target header before line {line_number}')
        rule = re.fullmatch(r'RULE\s+(\d+)\s*:\s*(\S.*)', line, re.I)
        if rule:
            rule_id = int(rule.group(1))
            if rule_id < 1 or rule_id in current['rules']:
                raise ValueError(f'Duplicate or invalid rule ID in {current["name"]}: {rule_id}')
            current['rules'][rule_id] = rule.group(2)
        elif not metadata.match(line):
            raise ValueError(f'Unexpected input at line {line_number}: {line[:120]}')
    if not targets or any(not entry['rules'] for entry in targets.values()):
        raise ValueError('Every target must contain at least one ground truth rule')
    return targets

gt_source = GT_FILE.read_text(encoding='utf-8-sig')
targets = parse_target_file(gt_source)
for entry in targets.values():
    print(f"{entry['name']:16s} {entry['type']:12s} {len(entry['rules']):4d} rules")
print(f'Total targets: {len(targets)}')
print(f"Total ground truth rules: {sum(len(entry['rules']) for entry in targets.values())}")


assert EXPECTED_TARGETS is None or len(targets) == EXPECTED_TARGETS, 'Unexpected target count'
assert EXPECTED_GT is None or sum(len(entry['rules']) for entry in targets.values()) == EXPECTED_GT, 'Unexpected ground truth count'

csrrwi           instruction     4 rules
fcvt.w.d         instruction     4 rules
sfence.vma       instruction    14 rules
vle32ff.v        instruction     5 rules
mseccfg          csr            73 rules
mstatus          csr           234 rules
satp             csr            45 rules
scounteren       csr            29 rules
Total targets: 8
Total ground truth rules: 408


## 3. Generation prompt and source

In [3]:
METHOD = 'DocAgent'
GENERATION_PROMPT = r'''You are an expert RISC-V ISA and privileged-architecture rule extractor.

Target:
Generate atomic architectural rules for the RISC-V `{target}`.

Source restriction:
Use only the provided RISC-V specification text and your pretrained knowledge of RISC-V architecture.
Do not browse the web.
Do not use external documents.
Do not use uploaded files other than the provided RISC-V specification text.
Do not cite sources.
Do not mention uncertainty.
The provided RISC-V specification text is the authoritative source. If your pretrained knowledge conflicts with the provided text, follow the provided text.

Input:
The input is the provided RISC-V specification text.
Use the text content as the main source for extracting rules.
You may use your pretrained RISC-V knowledge only to interpret terminology, connect related specification statements, and avoid missing explicit architectural behavior described in the text.

Reasoning requirement:
Before producing the final rules, reason step by step internally.
Internally perform the following reasoning steps:

1. Locate all specification content relevant to the target.
2. Recall the normative architectural semantics of the target.
3. Identify all architectural conditions that affect whether operations involving the target trap or execute.
4. Identify all architectural state conditions that affect the behavior of the target.
5. Identify all operand-dependent, value-dependent, field-dependent, or mode-dependent behaviors of the target.
6. Identify all privilege-dependent architectural behaviors of the target.
7. Identify all virtualization-dependent architectural behaviors of the target, if any are explicitly specified.
8. Identify all ordering, memory, translation, CSR, register, arithmetic, exception, and architectural-state effects of the target, if any are explicitly specified.
9. Split every independent condition -> outcome relationship into a separate atomic rule.
10. Remove duplicate, non-normative, non-testable, encoding-only, syntax-only, layout-only, and implementation-only statements.
11. Check that every remaining rule has an explicit architectural condition and an explicit architectural outcome.
12. Check that every remaining rule is testable using architectural observations such as trap versus no trap behavior, trap cause, CSR state, register state, privilege mode, virtualization state, memory state, arithmetic result, or address-translation behavior.

Do not output the reasoning.
Do not output intermediate analysis.
Output only the final rules.

Scope:
Extract rules only for the target.
Extract complete explicit normative architectural behavior rules for the target.
Include all architecturally testable privilege, trap, ordering, memory, translation, CSR, register, exception, arithmetic, operand-dependent, value-dependent, field-dependent, mode-dependent, and virtualization-related semantics that are explicitly specified for the target.
Do not include rules for related architectural items unless the specification explicitly defines their behavior as equivalent to, dependent on, or affecting the target.

Rule requirements:
Each rule must be a single testable statement.
Each rule must be atomic.
Each rule must express exactly one architectural behavior or one event -> outcome relationship.
Each rule must describe behavioral semantics, not naming, instruction encoding, assembly syntax, operand format, register layout, or commentary.
Each rule must describe architectural cause -> effect semantics.
Each rule must include both the architectural condition and the architectural outcome.
Each rule must be testable using architectural observations such as trap versus no trap behavior, trap cause, CSR state, register state, privilege mode, virtualization state, memory state, arithmetic result, or address-translation behavior.
Each rule must be explicit and normative.
Do not include implementation advice.
Do not include microarchitectural behavior unless the architecture gives an architecturally visible effect.
Do not include performance behavior.
Do not include recommendations.
Do not include examples.
Do not merge multiple independent behaviors into one rule.
If a behavior has multiple independent conditions or outcomes, split it into multiple rules.
If a rule depends on architectural state, privilege mode, virtualization state, operand value, memory state, CSR state, register state, exception behavior, arithmetic behavior, field value, mode value, or address-translation state, include that condition explicitly.
If a behavior is not architecturally observable or not testable, exclude it.

Completeness requirement:
Generate the complete set of explicit, normative, architecturally testable rules for the target from the provided RISC-V specification text.

Numbering:
Number the generated rules sequentially starting from 1.
Every rule must begin with the word `RULE`.

Output format:
Output only the rules.
Do not give an introduction.
Do not give headings.
Do not give bullet points.
Do not include explanations.
Do not include citations.
Do not include blank lines between rules.

Required rule format:
RULE <number>: <one atomic testable architectural rule>

RISC-V specification text:
{specification}'''
SPEC_FILE = Path('SPECIFICATION_ALL.txt')
assert SPEC_FILE.is_file(), f'Missing specification: {SPEC_FILE}'
specification = SPEC_FILE.read_text(encoding='utf-8-sig')
assert specification.strip(), 'Empty specification'
source_hash = hashlib.sha256(specification.encode('utf-8')).hexdigest()
assert CHUNK_CHARS + len(GENERATION_PROMPT) + 2000 < MAX_PROMPT_CHARS, 'Chunk configuration exceeds character budget'
print(f'Specification characters: {len(specification):,}')
print(f'Specification SHA256: {source_hash}')


Specification characters: 1,914,797
Specification SHA256: 78bfa481afefff6eb36683bd6f52e8cbe7095e0b52f707d55f584b938dfcaf56


## 4. Chunked rule generation with per-chunk checkpoints

In [4]:
def split_specification(text, max_chars=CHUNK_CHARS, overlap_chars=CHUNK_OVERLAP_CHARS):
    if not 0 <= overlap_chars < max_chars:
        raise ValueError('Invalid chunk settings')
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            boundary = text.rfind('\n', start + max_chars // 2, end)
            if boundary > start:
                end = boundary + 1
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(start + 1, end - overlap_chars)
    return chunks

def parse_generated_rules(text):
    value = text.strip()
    value = re.sub(r'^```(?:text|txt)?\s*', '', value, flags=re.I)
    value = re.sub(r'\s*```$', '', value).strip()
    if re.fullmatch(r'(?:NONE|NO RULES)\.?', value, re.I):
        return []
    pattern = re.compile(r'^\s*(?:[-*]\s*)?RULE\s+\d+\s*:\s*(\S.*)$', re.I)
    rules = [match.group(1).strip() for line in value.splitlines() if (match := pattern.fullmatch(line))]
    if not rules:
        raise ValueError(f'No RULE lines or NONE found: {value[:180]}')
    return rules

def format_rules(rules):
    return '\n'.join(f'RULE {i}: {rule}' for i, rule in enumerate(rules, 1))

def write_text(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(value.rstrip() + '\n', encoding='utf-8')
    temporary.replace(path)

def write_json(path, value):
    write_text(path, json.dumps(value, indent=2, ensure_ascii=False))

def run_target(target, target_type, load_target_gt, out, model):
    started = perf_counter()
    log_dir = out / 'llm_logs'
    checkpoint_dir = out / 'chunk_checkpoints'
    log_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    chunks = split_specification(specification)
    target_name = target.upper() if target_type == 'instruction' else target
    all_generated = []
    chunk_summaries = []

    write_json(out / 'run_config.json', {
        'target': target,
        'target_type': target_type,
        'model': model,
        'local': MODEL_SETTINGS[model]['local'],
        'method': METHOD,
        'source_sha256': source_hash,
        'total_chunks': len(chunks),
        'chunk_chars': CHUNK_CHARS,
        'chunk_overlap_chars': CHUNK_OVERLAP_CHARS,
        'evaluation': 'manual_only',
    })

    print(f'{target}: {len(chunks)} specification chunks')

    for chunk_id, chunk in enumerate(chunks, 1):
        prompt = GENERATION_PROMPT.replace('{target}', target_name)
        prompt = prompt.replace('{specification}', chunk)
        prompt += (
            f'\n\nFinal instruction for {target_name}: '
            'Extract testable rules from this excerpt only. '
            'If this excerpt contains no relevant architectural rules for the target, output exactly NONE. '
            'Otherwise output only RULE 1: ..., RULE 2: ..., etc., without headings or explanations.'
        )
        if len(prompt) > MAX_PROMPT_CHARS:
            raise ValueError(f'Chunk {chunk_id} exceeds character budget')

        checkpoint_file = checkpoint_dir / f'chunk_{chunk_id:04d}.json'
        prompt_sha256 = hashlib.sha256(prompt.encode('utf-8')).hexdigest()

        if checkpoint_file.is_file():
            saved = json.loads(checkpoint_file.read_text(encoding='utf-8'))
            if saved.get('prompt_sha256') != prompt_sha256 or saved.get('model') != model:
                raise ValueError(f'Checkpoint mismatch for {target}, chunk {chunk_id}')
            chunk_rules = saved['rules']
            if not isinstance(chunk_rules, list) or not all(isinstance(rule, str) for rule in chunk_rules):
                raise ValueError(f'Invalid checkpoint for {target}, chunk {chunk_id}')
            print(f'{target}: chunk {chunk_id}/{len(chunks)} resumed, rules={len(chunk_rules)}')
        else:
            write_text(log_dir / f'chunk_{chunk_id:04d}_prompt.txt', prompt)
            chunk_rules = None
            last_error = None

            output_limit = MAX_OUTPUT_TOKENS
            for attempt in range(1, LLM_ATTEMPTS + 1):
                try:
                    raw, finish_reason = call_llm(model, prompt, output_limit)
                    write_text(log_dir / f'chunk_{chunk_id:04d}_attempt_{attempt}.txt', raw)
                    write_json(log_dir / f'chunk_{chunk_id:04d}_attempt_{attempt}_meta.json', {
                        'output_limit': output_limit,
                        'finish_reason': finish_reason,
                    })
                    if finish_reason == 'length':
                        output_limit = min(output_limit * 2, MAX_RETRY_OUTPUT_TOKENS)
                        raise ValueError('LLM output reached the token limit')
                    chunk_rules = parse_generated_rules(raw)
                    break
                except Exception as exc:
                    last_error = exc
                    print(
                        f'{target}: chunk {chunk_id}/{len(chunks)}, attempt {attempt}/{LLM_ATTEMPTS} '
                        f'failed: {type(exc).__name__}: {str(exc)[:180]}'
                    )

            if chunk_rules is None:
                write_text(out / 'rule_partial.txt', format_rules(all_generated))
                raise RuntimeError(f'Generation failed at {target}, chunk {chunk_id}/{len(chunks)}') from last_error

            write_json(checkpoint_file, {
                'model': model,
                'target': target,
                'chunk_id': chunk_id,
                'prompt_sha256': prompt_sha256,
                'rules': chunk_rules,
            })
            print(f'{target}: chunk {chunk_id}/{len(chunks)} completed, rules={len(chunk_rules)}')

        all_generated.extend(chunk_rules)
        chunk_summaries.append({'chunk': chunk_id, 'generated': len(chunk_rules)})
        write_text(out / 'rule_partial.txt', format_rules(all_generated))
        write_json(out / 'chunk_progress.json', {
            'completed_chunks': chunk_id,
            'total_chunks': len(chunks),
            'raw_generated_so_far': len(all_generated),
        })

    if not all_generated:
        raise ValueError(f'No rules generated for {target} across {len(chunks)} chunks')

    write_text(out / 'rule.txt', format_rules(all_generated))
    unique_rules = []
    provenance = {}
    seen = {}
    for original_id, rule in enumerate(all_generated, 1):
        key = re.sub(r'\s+', ' ', rule).strip().lower()
        if key in seen:
            provenance[str(seen[key])].append(original_id)
            continue
        unique_id = len(unique_rules) + 1
        unique_rules.append(rule)
        seen[key] = unique_id
        provenance[str(unique_id)] = [original_id]

    write_text(out / 'rule_clean.txt', format_rules(unique_rules))
    write_json(out / 'rule_provenance.json', provenance)
    gt_text = load_target_gt()
    gt_count = len(targets[target]['rules'])
    write_text(out / 'ground_truth_snapshot.txt', gt_text)

    metrics = {
        'target': target,
        'ground_truth': gt_count,
        'raw_generated': len(all_generated),
        'unique_generated': len(unique_rules),
        'exact_duplicates_removed': len(all_generated) - len(unique_rules),
        'total_chunks': len(chunks),
        'model': model,
        'local': MODEL_SETTINGS[model]['local'],
        'method': METHOD,
        'source_sha256': source_hash,
        'ground_truth_sha256': hashlib.sha256(gt_text.encode('utf-8')).hexdigest(),
        'evaluation_status': 'NOT_EVALUATED',
        'manual_review_required': True,
        'elapsed_seconds': round(perf_counter() - started, 2),
    }
    write_json(out / 'metrics.json', metrics)
    write_text(out / 'FINAL_RESULT.txt', '\n'.join([
        'DOCAGENT GENERATION RESULT',
        f'Target: {target}',
        f'Model: {model}',
        f'Chunks: {len(chunks)}',
        f'Ground truth rules: {gt_count}',
        f'Raw generated rules: {len(all_generated)}',
        f'Unique generated rules: {len(unique_rules)}',
        'Semantic matching: NOT PERFORMED',
        'Correctness validation: NOT PERFORMED',
    ]))
    return metrics


## 5. Suite runner, automatic resume, and aggregation

In [5]:
def suite_configuration(model):
    return {
        'ground_truth_sha256': hashlib.sha256(gt_source.encode('utf-8')).hexdigest(),
        'targets': [
            {'name': entry['name'], 'type': entry['type'], 'ground_truth': len(entry['rules'])}
            for entry in targets.values()
        ],
        'method': METHOD,
        'model': model,
        'model_settings': MODEL_SETTINGS[model],
        'temperature': TEMPERATURE,
        'max_output_tokens': MAX_OUTPUT_TOKENS,
        'max_retry_output_tokens': MAX_RETRY_OUTPUT_TOKENS,
        'ollama_local_host': OLLAMA_LOCAL_HOST,
        'llm_attempts': LLM_ATTEMPTS,
        'max_prompt_chars': MAX_PROMPT_CHARS,
        'chunk_chars': CHUNK_CHARS,
        'chunk_overlap_chars': CHUNK_OVERLAP_CHARS,
        'source_sha256': source_hash,
        'generation_prompt_sha256': hashlib.sha256(GENERATION_PROMPT.encode('utf-8')).hexdigest(),
        'pipeline_version': 'docagent-chunked-checkpoint-local-v1',
    }

def find_incomplete_suite(model, current_config):
    model_dir = re.sub(r'[^A-Za-z0-9_.-]', '_', model)
    root = Path('results') / METHOD.lower() / model_dir
    for candidate in sorted(root.glob('*'), reverse=True):
        config_path = candidate / 'suite_config.json'
        status_path = candidate / 'suite_status.json'
        if not config_path.is_file() or not status_path.is_file():
            continue
        try:
            previous_config = json.loads(config_path.read_text(encoding='utf-8'))
            previous_status = json.loads(status_path.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        if previous_config != current_config:
            continue
        if all(previous_status.get(name, {}).get('status') == 'COMPLETED' for name in targets):
            continue
        return candidate.resolve()
    return None

def run_suite(model):
    config = suite_configuration(model)
    requested_resume = RESUME_SUITE_DIRS.get(model)
    if requested_resume:
        suite_out = Path(requested_resume).expanduser().resolve()
    elif AUTO_RESUME:
        suite_out = find_incomplete_suite(model, config)
    else:
        suite_out = None

    if suite_out is not None:
        previous_config = json.loads((suite_out / 'suite_config.json').read_text(encoding='utf-8'))
        if previous_config != config:
            raise ValueError('Resume configuration differs from the saved experiment')
        results = json.loads((suite_out / 'suite_status.json').read_text(encoding='utf-8'))
        print(f'Resuming suite: {suite_out}')
    else:
        suite_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
        model_dir = re.sub(r'[^A-Za-z0-9_.-]', '_', model)
        suite_out = (Path('results') / METHOD.lower() / model_dir / suite_id).resolve()
        suite_out.mkdir(parents=True, exist_ok=False)
        results = {}
        write_json(suite_out / 'suite_config.json', config)
        write_json(suite_out / 'suite_status.json', results)
        print(f'New suite: {suite_out}')

    for index, (name, entry) in enumerate(targets.items(), 1):
        previous = results.get(name, {})
        if previous.get('status') == 'COMPLETED':
            print(f'[{index}/{len(targets)}] {name}: already completed')
            continue

        previous_out = previous.get('out')
        if previous_out:
            out = Path(previous_out)
        else:
            attempt_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
            out = suite_out / name / attempt_id

        results[name] = {'status': 'RUNNING', 'out': str(out)}
        write_json(suite_out / 'suite_status.json', results)
        print(f'\n===== [{index}/{len(targets)}] {name} =====')

        def load_target_gt():
            return '\n'.join(f'RULE {i}: {rule}' for i, rule in sorted(entry['rules'].items()))

        started = perf_counter()
        try:
            metrics = run_target(name, entry['type'], load_target_gt, out, model)
            results[name] = {'status': 'COMPLETED', 'out': str(out), 'metrics': metrics}
        except Exception as exc:
            results[name] = {
                'status': 'FAILED',
                'out': str(out),
                'error': f'{type(exc).__name__}: {exc}',
                'elapsed_seconds': round(perf_counter() - started, 2),
            }
            print(f'{name} FAILED: {type(exc).__name__}: {exc}')
        finally:
            write_json(suite_out / 'suite_status.json', results)
    return suite_out

def aggregate(suite_out, model):
    results = json.loads((suite_out / 'suite_status.json').read_text(encoding='utf-8'))
    columns = ('Target', 'GT', 'Raw', 'Unique', 'Status')
    row_format = '{:<16} {:>5} {:>8} {:>8}  {}'
    lines = [
        f'SUPER FINAL RESULT - {METHOD} - {model}',
        '',
        row_format.format(*columns),
        '-' * 60,
    ]
    total_gt = 0
    total_raw = 0
    total_unique = 0
    completed = 0
    all_rules = []

    for name, entry in targets.items():
        item = results.get(name, {'status': 'NOT_RUN'})
        if item['status'] == 'COMPLETED':
            metrics = item['metrics']
            assert metrics['ground_truth'] == len(entry['rules'])
            completed += 1
            total_gt += len(entry['rules'])
            total_raw += metrics['raw_generated']
            total_unique += metrics['unique_generated']
            lines.append(row_format.format(name, len(entry['rules']), metrics['raw_generated'], metrics['unique_generated'], 'COMPLETED'))
            cleaned = (Path(item['out']) / 'rule_clean.txt').read_text(encoding='utf-8').strip()
            header = 'INSTRUCTION' if entry['type'] == 'instruction' else 'REGISTER'
            all_rules.append(f'{header}: {name}\n{cleaned}')
        else:
            lines.append(row_format.format(name, len(entry['rules']), '-', '-', item['status']))

    full_gt = sum(len(entry['rules']) for entry in targets.values())
    lines.extend([
        '-' * 60,
        row_format.format('TOTAL COMPLETED', total_gt, total_raw, total_unique, f'{completed}/{len(targets)}'),
        '',
        f'Total input GT rules: {full_gt}',
        f'Completed targets: {completed}/{len(targets)}',
        f'Total raw generated rules: {total_raw}',
        f'Total unique generated rules: {total_unique}',
        '',
        'Semantic matching: NOT PERFORMED',
        'Correctness validation: NOT PERFORMED',
        'Precision: NOT AVAILABLE',
        'Recall: NOT AVAILABLE',
        'Manual verification required.',
    ])
    if completed != len(targets):
        lines.extend(['', 'INCOMPLETE SUITE'])

    write_text(suite_out / 'SUPER_FINAL_RESULT.txt', '\n'.join(lines))
    write_text(suite_out / 'GENERATED_RULES_ALL.txt', '\n\n'.join(all_rules))
    summary = {
        'model': model,
        'method': METHOD,
        'ground_truth': total_gt,
        'total_input_gt': full_gt,
        'raw_generated': total_raw,
        'unique_generated': total_unique,
        'completed_targets': completed,
        'total_targets': len(targets),
        'complete': completed == len(targets),
        'evaluation_status': 'NOT_EVALUATED',
        'manual_review_required': True,
        'source_sha256': source_hash,
    }
    write_json(suite_out / 'SUPER_FINAL_METRICS.json', summary)
    print('\n'.join(lines))
    print(f'Saved to: {suite_out}')
    return summary


## 6. Run Gemma 4 E2B and 12B locally

In [6]:
suite_dirs = {}
summaries = {}
for model in RUN_MODELS:
    print(f'\n===== {METHOD}: {model} (Local) =====')
    print(f'Specification chunks per target: {len(split_specification(specification))}')
    suite_dirs[model] = run_suite(model)
    summaries[model] = aggregate(suite_dirs[model], model)

print('\nMODEL COMPARISON - DocAgent Local')
for model in RUN_MODELS:
    metrics = summaries[model]
    print(
        f'{model:16s} GT={metrics["total_input_gt"]} '
        f'raw={metrics["raw_generated"]} unique={metrics["unique_generated"]} '
        f'completed={metrics["completed_targets"]}/{metrics["total_targets"]}'
    )


===== DocAgent: gemma4:e2b (Local) =====
Specification chunks per target: 25
New suite: /mnt/data2/new/first_project/project_1/oculus_isa/9_EXPERIMENT/34_new_study/sens_spec/4_doc_2/results/docagent/gemma4_e2b/20261123_213532_393601

===== [1/8] csrrwi =====
csrrwi: 25 specification chunks
csrrwi: chunk 1/25 completed, rules=0
csrrwi: chunk 2/25 completed, rules=4
csrrwi: chunk 3/25 completed, rules=1
csrrwi: chunk 4/25 completed, rules=1
csrrwi: chunk 5/25 completed, rules=0
csrrwi: chunk 6/25 completed, rules=0
csrrwi: chunk 7/25 completed, rules=0
csrrwi: chunk 8/25 completed, rules=0
csrrwi: chunk 9/25 completed, rules=10
csrrwi: chunk 10/25 completed, rules=0
csrrwi: chunk 11/25 completed, rules=0
csrrwi: chunk 12/25 completed, rules=0
csrrwi: chunk 13/25 completed, rules=7
csrrwi: chunk 14/25 completed, rules=0
csrrwi: chunk 15/25 completed, rules=0
csrrwi: chunk 16/25 completed, rules=1
csrrwi: chunk 17/25 completed, rules=5
csrrwi: chunk 18/25 completed, rules=0
csrrwi: chunk 

## 7. View suite status and generated rules for both models

In [7]:
for model, suite_dir in suite_dirs.items():
    print(f'\nModel: {model}')
    print(f'Suite: {suite_dir}')
    status = json.loads((suite_dir / 'suite_status.json').read_text(encoding='utf-8'))
    for name, entry in status.items():
        checkpoint_path = Path(entry['out']) / 'chunk_checkpoints'
        completed_chunks = len(list(checkpoint_path.glob('chunk_*.json')))
        print(f'{name:16s} {entry["status"]:10s} checkpoint chunks: {completed_chunks}')
    print(f'Generated rules: {suite_dir / "GENERATED_RULES_ALL.txt"}')


Model: gemma4:e2b
Suite: /mnt/data2/new/first_project/project_1/oculus_isa/9_EXPERIMENT/34_new_study/sens_spec/4_doc_2/results/docagent/gemma4_e2b/20261123_213532_393601
csrrwi           COMPLETED  checkpoint chunks: 25
fcvt.w.d         COMPLETED  checkpoint chunks: 25
sfence.vma       COMPLETED  checkpoint chunks: 25
vle32ff.v        COMPLETED  checkpoint chunks: 25
mseccfg          COMPLETED  checkpoint chunks: 25
mstatus          COMPLETED  checkpoint chunks: 25
satp             COMPLETED  checkpoint chunks: 25
scounteren       COMPLETED  checkpoint chunks: 25
Generated rules: /mnt/data2/new/first_project/project_1/oculus_isa/9_EXPERIMENT/34_new_study/sens_spec/4_doc_2/results/docagent/gemma4_e2b/20261123_213532_393601/GENERATED_RULES_ALL.txt

Model: gemma4:12b
Suite: /mnt/data2/new/first_project/project_1/oculus_isa/9_EXPERIMENT/34_new_study/sens_spec/4_doc_2/results/docagent/gemma4_12b/20261123_215537_973587
csrrwi           COMPLETED  checkpoint chunks: 25
fcvt.w.d         COMPL